In [ ]:
# ==============================================================================
# BLOQUE 1: INSTALACIONES
# ==============================================================================
%pip install -q -U openai-whisper google-api-python-client google-auth-httplib2 google-auth-oauthlib
!apt-get update -qq && apt-get install -y ffmpeg -qq

# ==============================================================================
# BLOQUE 2: IMPORTACIONES Y CONFIGURACIÓN INICIAL
# ==============================================================================
import os, subprocess, re, logging, io, tempfile, sys, time, zipfile, importlib
from pathlib import Path
import torch
torch._utils = importlib.import_module("torch._utils")
import whisper as whisper_ai

# Google Colab y APIs
from google.colab import auth, files
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from googleapiclient.errors import HttpError

# Configuración de Logging
logging.basicConfig(
    filename='transcription_process.log',
    level=logging.INFO,
    format='%(asctime)s:%(levelname)s:%(message)s',
    filemode='w'
)

def log_info(msg: str):
    print(msg, flush=True)
    logging.info(msg)

def log_error(msg: str, exc_info=False):
    print(f"❌ ERROR: {msg}")
    logging.error(msg, exc_info=exc_info)

# Verificar Dispositivo. Whisper se carga al empezar a transcribir.
device = "cuda" if torch.cuda.is_available() else "cpu"
log_info(f"🧩 PyTorch: {torch.__version__} | CUDA: {torch.version.cuda} | Archivo: {getattr(torch, '__file__', 'N/A')}")
log_info(f"⚙️ Dispositivo detectado para Whisper: {device}")

whisper_model_instance = None

def load_whisper_model_safely(model_name: str, device: str):
    """Carga Whisper evitando un fallo ocasional de PyTorch en Colab con torch._utils."""
    try:
        return whisper_ai.load_model(model_name, device=device)
    except AttributeError as e:
        if "torch" in str(e) and "_utils" in str(e):
            log_info("⚠️ Ajustando PyTorch de Colab y reintentando cargar Whisper...")
            torch._utils = importlib.import_module("torch._utils")
            return whisper_ai.load_model(model_name, device=device)
        raise

def get_whisper_model():
    """Carga el modelo una sola vez, justo antes de la primera transcripción."""
    global whisper_model_instance
    if whisper_model_instance is None:
        model_name = "large-v3-turbo" if device == "cuda" else "small"
        log_info(f"⏳ Cargando el modelo Whisper '{model_name}'... (esto puede tardar un momento)")
        whisper_model_instance = load_whisper_model_safely(model_name, device)
        whisper_model_instance.eval()
        log_info("✅ Modelo Whisper cargado exitosamente.")
    return whisper_model_instance

# ==============================================================================
# BLOQUE 3: FUNCIONES AUXILIARES
# ==============================================================================
def sanitize_filename(name: str) -> str:
    """Limpia un nombre de archivo reemplazando caracteres no permitidos."""
    cleaned = re.sub(r'[\\/*?:"<>|]', "_", name.replace(" ", "_"))
    return cleaned.strip("._") or "archivo"

def authenticate_google() -> tuple:
    """Autentica con Google Drive y Docs APIs solo cuando se necesita."""
    log_info("🔑 Iniciando autenticación con Google (Drive y Docs)...")
    try:
        auth.authenticate_user()
        creds, _ = google.auth.default(scopes=[
            "https://www.googleapis.com/auth/drive",
            "https://www.googleapis.com/auth/documents",
        ])
        drive_service = build("drive", "v3", credentials=creds, cache_discovery=False)
        docs_service = build("docs", "v1", credentials=creds, cache_discovery=False)
        log_info("✅ Autenticación exitosa.")
        return drive_service, docs_service
    except Exception as e:
        log_error(f"Fallo en autenticación de Google: {e}", exc_info=True)
        return None, None

def list_drive_files(drive_service, folder_id: str) -> list:
    """Lista archivos de video/audio en una carpeta de Google Drive."""
    log_info(f"📂 Buscando archivos en la carpeta ID: {folder_id}...")
    items_list = []
    page_token = None
    query = f"'{folder_id}' in parents and trashed=false and (mimeType contains 'video/' or mimeType contains 'audio/' or mimeType = 'application/octet-stream')"

    try:
        while True:
            response = drive_service.files().list(
                q=query, spaces="drive", fields="nextPageToken, files(id, name, mimeType, size)",
                orderBy="name", pageSize=100, pageToken=page_token
            ).execute()
            items_list.extend(response.get("files", []))
            page_token = response.get("nextPageToken")
            if not page_token: break

        if items_list:
            print("\nArchivos encontrados:")
            for idx, f in enumerate(items_list, 1):
                size_mb = f"{int(f.get('size', 0)) / (1024*1024):.2f} MB" if f.get('size', '').isdigit() else "N/A"
                print(f"  [{idx}] {f['name']} ({size_mb})")
        else:
            log_info("No se encontraron audios o videos en esa carpeta.")
        return items_list
    except Exception as e:
        log_error(f"Error listando archivos: {e}")
        return []

def select_indices(items_list: list) -> list:
    """Permite elegir uno, varios o todos los archivos listados."""
    if len(items_list) == 1:
        log_info("✅ Hay un solo archivo disponible; se seleccionará automáticamente.")
        return [0]

    selection = input("\n🔢 Ingresa los NÚMEROS a transcribir (ej: 1,3) o escribe 'todos': ").strip().lower()
    if selection == 'todos':
        return list(range(len(items_list)))

    try:
        indices = [int(x.strip()) - 1 for x in selection.split(",")]
        return [idx for idx in indices if 0 <= idx < len(items_list)]
    except Exception:
        log_error("Selección inválida.")
        return []

def upload_local_files(temp_dir: Path) -> list:
    """Sube uno o varios archivos desde el computador al entorno de Colab."""
    print("\n📤 Selecciona uno o varios audios/videos desde tu computador.")
    uploaded = files.upload()
    local_files = []

    for original_name, content in uploaded.items():
        safe_name = sanitize_filename(original_name)
        local_path = temp_dir / safe_name
        local_path.write_bytes(content)
        local_files.append({
            "source": "local",
            "name": original_name,
            "path": local_path,
            "size": len(content),
        })

    if local_files:
        print("\nArchivos subidos:")
        for idx, f in enumerate(local_files, 1):
            print(f"  [{idx}] {f['name']} ({f['size'] / (1024*1024):.2f} MB)")
    else:
        log_info("No se subió ningún archivo.")

    return local_files

def extract_audio_if_video(media_path: Path, temp_dir: Path) -> Path:
    """Si el archivo es video, extrae el audio a MP3. Si ya es audio, lo devuelve igual."""
    video_exts = (".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv", ".wmv", ".m4v")
    if media_path.suffix.lower() not in video_exts:
        return media_path

    log_info("🎬 Video detectado. Extrayendo audio...")
    mp3_path = temp_dir / f"{media_path.stem}.mp3"
    cmd = ["ffmpeg", "-i", str(media_path), "-vn", "-acodec", "libmp3lame", "-ar", "44100", "-ac", "2", "-q:a", "2", "-y", str(mp3_path)]
    subprocess.run(cmd, check=True, capture_output=True)
    return mp3_path

def download_and_prepare_audio(drive_service, file_id: str, original_filename: str, temp_dir: Path) -> Path:
    """Descarga un archivo de Drive y, si es video, extrae el audio con ffmpeg."""
    log_info(f"⬇️ Descargando: '{original_filename}'...")
    download_path = temp_dir / sanitize_filename(original_filename)

    try:
        request = drive_service.files().get_media(fileId=file_id)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

        download_path.write_bytes(fh.getvalue())
        return extract_audio_if_video(download_path, temp_dir)
    except Exception as e:
        log_error(f"Error descargando/convirtiendo '{original_filename}': {e}")
        return None

def prepare_local_audio(local_path: Path, temp_dir: Path) -> Path:
    """Prepara un archivo local subido a Colab para transcripción."""
    try:
        log_info(f"📁 Preparando archivo local: '{local_path.name}'...")
        return extract_audio_if_video(local_path, temp_dir)
    except Exception as e:
        log_error(f"Error preparando archivo local '{local_path.name}': {e}")
        return None

def get_audio_duration(file_path: Path) -> float:
    """Obtiene la duración del audio en segundos usando ffprobe."""
    try:
        cmd = ["ffprobe", "-i", str(file_path), "-show_entries", "format=duration", "-v", "quiet", "-of", "csv=p=0"]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return float(result.stdout.strip())
    except Exception as e:
        log_error(f"Error obteniendo duración de audio: {e}")
        return 0.0

def transcribe_audio(audio_path: Path) -> tuple:
    """Pasa el audio por Whisper, lo transcribe y calcula los tiempos."""
    log_info(f"🗣️ Transcribiendo: {audio_path.name}...")

    audio_seconds = get_audio_duration(audio_path)
    start_time = time.time()

    try:
        model = get_whisper_model()
        log_info("🔄 Whisper está transcribiendo ahora. Esto puede tardar varios minutos según la duración del archivo...")
        with torch.no_grad():
            result = model.transcribe(str(audio_path), language="es", fp16=torch.cuda.is_available())
        text = result.get("text", "").strip()

        elapsed_seconds = time.time() - start_time
        rtf = elapsed_seconds / audio_seconds if audio_seconds > 0 else 0

        log_info(f"✅ Transcripción completada ({len(text)} caracteres).")
        log_info(f"⏱️ Audio: {audio_seconds:.2f}s | Proceso: {elapsed_seconds:.2f}s | RTF: {rtf:.4f}")

        return text, audio_seconds, elapsed_seconds, rtf
    except Exception as e:
        log_error(f"Error en transcripción: {e}")
        return None, 0.0, 0.0, 0.0

def save_to_gdocs(text: str, title: str, folder_id: str, docs_service, drive_service):
    """Guarda el texto en un nuevo Google Doc."""
    doc_title = f"Transcripción_{sanitize_filename(title)}"
    log_info(f"📄 Guardando en Google Docs: '{doc_title}'...")
    try:
        doc = docs_service.documents().create(body={"title": doc_title}).execute()
        doc_id = doc["documentId"]

        docs_service.documents().batchUpdate(
            documentId=doc_id,
            body={"requests": [{"insertText": {"location": {"index": 1}, "text": text}}]}
        ).execute()

        if folder_id:
            file_meta = drive_service.files().get(fileId=doc_id, fields="parents").execute()
            prev_parents = ",".join(file_meta.get("parents", []))
            drive_service.files().update(
                fileId=doc_id, addParents=folder_id, removeParents=prev_parents, fields="id"
            ).execute()

        doc_url = f"https://docs.google.com/document/d/{doc_id}/edit"
        log_info(f"✅ Doc guardado: {doc_url}")
        return doc_url
    except Exception as e:
        log_error(f"Error guardando en Docs: {e}")
        return None

def save_to_txt(text: str, title: str, output_dir: Path) -> Path:
    """Guarda la transcripción como archivo TXT descargable."""
    output_dir.mkdir(parents=True, exist_ok=True)
    txt_path = output_dir / f"Transcripcion_{sanitize_filename(title)}.txt"
    txt_path.write_text(text, encoding="utf-8")
    log_info(f"✅ TXT guardado: {txt_path.name}")
    return txt_path

def download_txt_outputs(txt_paths: list):
    """Descarga uno o varios TXT al computador del usuario."""
    if not txt_paths:
        return

    if len(txt_paths) == 1:
        files.download(str(txt_paths[0]))
        return

    zip_path = txt_paths[0].parent / "transcripciones_txt.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for txt_path in txt_paths:
            zf.write(txt_path, arcname=txt_path.name)
    log_info(f"📦 Descargando ZIP con {len(txt_paths)} transcripciones TXT...")
    files.download(str(zip_path))

def choose_source_mode() -> str:
    print("\n📥 ¿Desde dónde quieres tomar los audios/videos?")
    print("1. Carpeta de Google Drive")
    print("2. Subir archivos desde mi computador")
    option = input("Elige (1 o 2): ").strip()
    return "drive" if option == "1" else "local" if option == "2" else ""

def choose_output_mode() -> str:
    print("\n💾 ¿Cómo quieres guardar las transcripciones?")
    print("1. Google Docs")
    print("2. Archivos TXT descargables")
    print("3. Ambos")
    option = input("Elige (1, 2 o 3): ").strip()
    return {"1": "gdocs", "2": "txt", "3": "both"}.get(option, "")

def choose_gdocs_folder(source_mode: str, source_folder_id: str = None):
    if source_mode == "drive":
        print("\n📄 ¿Dónde quieres guardar los Google Docs resultantes?")
        print("1. En la MISMA carpeta de origen")
        print("2. En OTRA carpeta (pedirá ID)")
        print("3. En la RAÍZ de mi Google Drive")
        save_opt = input("Elige (1, 2 o 3): ").strip()
        if save_opt == "1":
            return source_folder_id
        if save_opt == "2":
            return input("Pega el ID de la carpeta de destino: ").strip()
        return None

    print("\n📄 ¿Dónde quieres guardar los Google Docs resultantes?")
    print("1. En la RAÍZ de mi Google Drive")
    print("2. En una carpeta específica (pedirá ID)")
    save_opt = input("Elige (1 o 2): ").strip()
    if save_opt == "2":
        return input("Pega el ID de la carpeta de destino: ").strip()
    return None

# ==============================================================================
# BLOQUE 4: FLUJO PRINCIPAL
# ==============================================================================
processing_stats = []

def main():
    global processing_stats
    processing_stats = []
    txt_outputs = []
    doc_outputs = []

    print("\n" + "="*60)
    print("🎙️ ASISTENTE DE TRANSCRIPCIÓN (WHISPER)")
    print("="*60)

    source_mode = choose_source_mode()
    if not source_mode:
        log_error("Opción de entrada inválida. Cancelando.")
        return

    output_mode = choose_output_mode()
    if not output_mode:
        log_error("Opción de guardado inválida. Cancelando.")
        return

    needs_google = source_mode == "drive" or output_mode in ("gdocs", "both")
    drive_service, docs_service = (None, None)
    if needs_google:
        drive_service, docs_service = authenticate_google()
        if not drive_service:
            return

    dest_folder_id = None
    source_folder_id = None

    with tempfile.TemporaryDirectory() as temp_dir:
        temp_path = Path(temp_dir)
        output_dir = Path("/content/transcripciones_txt") if Path("/content").exists() else Path("transcripciones_txt")

        if source_mode == "drive":
            source_folder_id = input("\n📁 Pega el ID de la carpeta de Drive donde están tus audios/videos: ").strip()
            if not source_folder_id:
                log_error("No se ingresó ID. Cancelando.")
                return

            files_list = list_drive_files(drive_service, source_folder_id)
            if not files_list:
                return
            indices = select_indices(files_list)
            if not indices:
                log_error("Ningún archivo válido seleccionado.")
                return
            selected_items = [{"source": "drive", **files_list[idx]} for idx in indices]
        else:
            files_list = upload_local_files(temp_path)
            if not files_list:
                return
            indices = select_indices(files_list)
            if not indices:
                log_error("Ningún archivo válido seleccionado.")
                return
            selected_items = [files_list[idx] for idx in indices]

        if output_mode in ("gdocs", "both"):
            dest_folder_id = choose_gdocs_folder(source_mode, source_folder_id)

        for f_info in selected_items:
            print("\n" + "-"*50)
            original_name = f_info["name"]

            if f_info["source"] == "drive":
                audio_path = download_and_prepare_audio(drive_service, f_info["id"], original_name, temp_path)
            else:
                audio_path = prepare_local_audio(Path(f_info["path"]), temp_path)
            if not audio_path:
                continue

            text, a_sec, e_sec, rtf = transcribe_audio(audio_path)
            if not text:
                continue

            processing_stats.append({
                "name": original_name,
                "audio_seconds": a_sec,
                "elapsed_seconds": e_sec,
                "rtf": rtf
            })

            title = Path(original_name).stem
            if output_mode in ("gdocs", "both"):
                doc_url = save_to_gdocs(text, title, dest_folder_id, docs_service, drive_service)
                if doc_url:
                    doc_outputs.append(doc_url)
            if output_mode in ("txt", "both"):
                txt_outputs.append(save_to_txt(text, title, output_dir))

            try:
                if audio_path.exists():
                    os.remove(audio_path)
            except Exception:
                pass

    if txt_outputs:
        download_txt_outputs(txt_outputs)

    print("\n🎉 ¡PROCESO COMPLETADO!")
    if doc_outputs:
        print("\n📄 Google Docs creados:")
        for url in doc_outputs:
            print(f"- {url}")
    if txt_outputs:
        print("\n📝 TXT generados:")
        for path in txt_outputs:
            print(f"- {path.name}")
    print("👉 Revisa los archivos descargados o tu Google Drive según la opción elegida.")

# Ejecutar el script
if __name__ == "__main__":
    main()


